# Bronze Layer Transformation

This notebook processes data from the landing layer into the bronze layer by:
* Adding ingestion timestamp metadata (`date_ingestion`)
* Standardizing data types for consistency
* Preparing clean, typed data for downstream silver layer transformations

**Source**: kyc.landing.* tables
**Target**: kyc.bronze.* tables

**Bronze Layer Purpose**: Raw data with minimal transformations - adds metadata and enforces schema

## Setup

Import required PySpark libraries for data transformation.

In [0]:
# Import PySpark types for schema definitions and type casting
from pyspark.sql.types import *

# Import PySpark functions for data transformations
from pyspark.sql.functions import *

In [0]:
# ============================================================================
# READ FROM LANDING LAYER
# ============================================================================

brze_dim_carburant_df = spark.read.table("kyc.landing.raw_dim_carburant")
brze_dim_geo_df       = spark.read.table("kyc.landing.raw_dim_geo")
brze_dim_service_df   = spark.read.table("kyc.landing.raw_dim_service")
brze_dim_station_df   = spark.read.table("kyc.landing.raw_dim_station")
brze_fait_prix_df     = spark.read.table("kyc.landing.raw_fait_prix")
brze_fait_rupture_df  = spark.read.table("kyc.landing.raw_fait_rupture")

In [0]:
# ============================================================================
# BRONZE DIMENSIONS  TRANSFORMATIONS
# ============================================================================

brze_dim_carburant_df = brze_dim_carburant_df\
            .withColumn("date_ingestion", now())\
            .withColumn("id", col("id").cast(IntegerType()))\
            .select("id","nom","date_ingestion")

brze_dim_geo_df = brze_dim_geo_df\
            .withColumn("date_ingestion", now())\
            .select(
                "code_region",
                "region",
                "code_departement",
                "departement",
                "date_ingestion"
        )

brze_dim_service_df = brze_dim_service_df\
                      .withColumn("date_ingestion", now())\
                      .select("station_id", "service", "date_ingestion")

brze_dim_station_df = brze_dim_station_df\
                      .withColumn("date_ingestion", now())\
                      .select(
                          "station_id",
                          "adresse",
                          "ville",
                          "code_postal",
                          "code_departement",
                          "latitude",
                          "longitude",
                          "service",
                          "code_region",
                          "date_ingestion"
                )

In [ ]:

# ============================================================================
# BRONZE FACTS  TRANSFORMATIONS
# ============================================================================
brze_fait_prix_df = brze_fait_prix_df\
                    .withColumn("date_ingestion", now())\
                    .withColumn("carburant_id", col("carburant_id").cast(IntegerType()))\
                    .withColumn("date_maj", expr("try_cast(date_maj AS TIMESTAMP)"))\
                    .withColumn("prix", col("date_ingestion").cast(IntegerType()))\
                    .select("station_id", "carburant_id", "date_maj", "prix", "date_ingestion")
                        
brze_fait_rupture_df = brze_fait_rupture_df\
                        .withColumn("date_ingestion", now())\
                        .withColumn("id_carburant", col("id_carburant").cast(IntegerType()))\
                        .withColumn("debut_rupture", expr("try_cast(debut_rupture AS TIMESTAMP)"))\
                        .withColumn("fin_rupture", expr("try_cast(fin_rupture AS TIMESTAMP)"))\
                        .select("station_id", "id_carburant", "debut_rupture", "fin_rupture", "type_rupture", "date_ingestion")

                        


In [0]:
# ============================================================================
# PERSIST TO BRONZE LAYER
# ============================================================================

brze_dim_carburant_df\
    .write\
    .format("delta")\
    .mode("append")\
    .saveAsTable("kyc.bronze.brze_dim_carburant")

brze_dim_geo_df\
    .write\
    .format("delta")\
    .mode('append')\
    .saveAsTable("kyc.bronze.brze_dim_geo")

brze_dim_service_df\
    .write\
    .mode("append")\
    .format("delta")\
    .saveAsTable("kyc.bronze.brze_dim_service")

brze_dim_station_df\
    .write\
    .format("delta")\
    .mode("append")\
    .saveAsTable("kyc.bronze.brze_dim_station")

brze_fait_prix_df\
    .write\
    .format("delta")\
    .mode("append")\
    .saveAsTable("kyc.bronze.brze_fait_prix")

brze_fait_rupture_df\
    .write\
    .format("delta")\
    .mode("append")\
    .saveAsTable("kyc.bronze.brze_fait_rupture")
